# HAIKUTHON — jina-clip-v2 zero-shot 평가 (Kaggle, GPU T4 x2 중 cuda:0만 사용)
- 모델: `jinaai/jina-clip-v2` (텍스트 Jina-XLM-RoBERTa 최대 8192 토큰, 비전 EVA02-L/14 @512px, 출력 1024차원 Matryoshka)
- 검증셋: 팀 공통 규칙 `RandomState(42).permutation` 앞 500개
- 측정: Recall@1 / NDCG@5 / 최종 점수, 이미지 인코딩 속도, 토큰 길이, fp16 가중치 용량
- 결과는 `/kaggle/working/jina_clip_v2/` 에 저장되며, 세션이 끊겨도 이어서 실행할 수 있습니다.
- `# [확인 필요]` 주석은 공식 문서로 100% 확인하지 못한 부분입니다.

In [1]:
# 셀 1. 패키지 확인 (Kaggle 기본 이미지에 없는 것만 설치)
import importlib
import subprocess
import sys


def ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
    except ImportError:
        print(f"installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


# jina-clip-v2 원격 코드가 사용하는 패키지 (공식 README 기준: einops, timm)
for p in ["einops", "timm"]:
    ensure(p)
# xformers 는 비전 타워(EVA02) 가속용 선택 사항이라 설치하지 않습니다. # [확인 필요] 없으면 기본 attention 으로 동작

import torch
import transformers
import datasets

print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("GPU:", torch.cuda.get_device_name(0))
# 주의: 채점 서버는 transformers 5.22 입니다. 여기서 돌아가도 서버에서 원격 코드가 깨질 수 있으니 버전을 꼭 기록하세요.


torch 2.11.0+cu130 | transformers 5.20.2 | datasets 4.8.5
GPU: Tesla T4


In [2]:
# 셀 2. 설정
import json
import math
import os
import time

import numpy as np

MODEL_ID = "jinaai/jina-clip-v2"
DATASET_ID = "KimWooJin/26-haikuthon-train"
WORK = "/kaggle/working"
OUT = os.path.join(WORK, "jina_clip_v2")
os.makedirs(OUT, exist_ok=True)

DEVICE = "cuda:0"          # T4 2장 중 1장만 사용
DTYPE = torch.float16      # T4 는 bf16 미지원 → fp16
IMG_BS = 32                # T4 16GB 기준. OOM 나면 16 으로
TXT_BS = 64
NUM_WORKERS = 4            # 채점 서버도 4 vCPU 이므로 동일 조건으로 측정
TEXT_TASKS = ["retrieval.query", None]   # 텍스트 LoRA 어댑터 비교: 검색 쿼리용 vs 미지정
TRUNC_DIMS = [1024, 512, 256]            # Matryoshka 차원 축소 비교 (저장된 임베딩에서 잘라서 계산)


In [3]:
# 셀 3. 데이터셋 로드 및 구조 출력
from datasets import Image as HFImage
from datasets import load_dataset

dd = load_dataset(DATASET_ID)
split = "train" if "train" in dd else list(dd.keys())[0]
ds = dd[split]
print(dd)
print("features:", ds.features)

# 이미지 컬럼 / 텍스트 컬럼 자동 탐지 (텍스트는 평균 길이가 가장 긴 문자열 컬럼)
IMG_COL = next(k for k, v in ds.features.items() if isinstance(v, HFImage))
str_cols = [k for k, v in ds.features.items() if getattr(v, "dtype", None) == "string"]
TXT_COL = max(str_cols, key=lambda k: np.mean([len(x) for x in ds[:20][k]]))
print(f"\n이미지 컬럼: {IMG_COL} | 텍스트 컬럼: {TXT_COL} | 총 {len(ds)}쌍\n")
for i in range(3):
    ex = ds[i]
    print(f"[{i}] size={ex[IMG_COL].size} mode={ex[IMG_COL].mode}")
    print("    ", ex[TXT_COL], "\n")


README.md:   0%|          | 0.00/4.49k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/4001 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/4001 [00:00<?, ?it/s]

images/0004.png: reconstructing file:   0%|          |  0.00B /  893kB            

images/0011.png: reconstructing file:   0%|          |  0.00B /  589kB            

images/0013.png: reconstructing file:   0%|          |  0.00B / 1.27MB            

... (7,999 outputs left out)

images/3999.png: downloading bytes:           |  0.00B            

metadata.jsonl:   0%|          | 0.00/1.01M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'id', 'prompts'],
        num_rows: 4000
    })
})
features: {'image': Image(mode=None, decode=True), 'id': Value('string'), 'prompts': Value('string')}

이미지 컬럼: image | 텍스트 컬럼: prompts | 총 4000쌍

[0] size=(768, 768) mode=RGB
     One person is lying on an airport apron. One scarf is solid-colored and gray. The person is wearing the gray scarf. One scarf is polka-dotted and orange. The person is above the orange scarf. The scene is a photograph from a front view with side lighting. 

[1] size=(768, 768) mode=RGB
     One square yellow drinking glass is above one white ear of corn and above one orange pumpkin in a kitchen. The scene is a colored-pencil drawing in side view with warm indoor lighting. 

[2] size=(768, 768) mode=RGB
     One brown shelf is to the left of one beige cookie. One green kettle is present. The scene is in a dining room. It is rendered as a flat illustration from a front view with warm indoor lighting

In [4]:
# 셀 4. 검증셋 분리 (팀 공통 — 수정 금지)
idx = np.random.RandomState(42).permutation(len(ds))
val_idx = sorted(idx[:500].tolist())
train_idx = sorted(idx[500:].tolist())
with open(os.path.join(WORK, "val_idx.json"), "w") as f:
    json.dump(val_idx, f)
print("val 앞 10개:", val_idx[:10])
print("train", len(train_idx), "| val", len(val_idx))

val_ds = ds.select(val_idx)
val_texts = val_ds[TXT_COL]


val 앞 10개: [8, 12, 14, 17, 23, 26, 32, 33, 44, 45]
train 3500 | val 500


In [5]:
# 셀 5. 모델 로드 + fp16 가중치 용량
from transformers import AutoModel, AutoTokenizer

# ── transformers 5.x 호환 패치 (로컬 kaggle/jina_compat.py 와 동일) ──────────────────────────────
# 문제 1) 원격 코드가 transformers 5.x 에서 삭제된 clip_loss 를 import → 같은 이름의 함수를 넣어 줌 (학습용, 추론엔 안 쓰임)
# 문제 2) 5.x 는 meta 장치에서 모델을 만든 뒤 가중치만 채움 → 가중치 파일에 없는 '비저장 버퍼'
#         (텍스트/이미지 RoPE 위치 주파수, LoRA mask)가 쓰레기 값으로 남아 에러 없이 엉터리 임베딩이 나옴
#         → 원래 공식대로 다시 계산해서 덮어씀
# 그 밖에: jina config 는 torch_dtype 을 문자열로만 받음 → fp32 로 불러온 뒤 .half()
import transformers.models.clip.modeling_clip as _mc

if not hasattr(_mc, "clip_loss"):
    def _clip_loss(similarity):
        t = torch.arange(len(similarity), device=similarity.device)
        return (torch.nn.functional.cross_entropy(similarity, t)
                + torch.nn.functional.cross_entropy(similarity.t(), t)) / 2.0
    _mc.clip_loss = _clip_loss


@torch.no_grad()
def fix_buffers(model):
    vc = model.config.vision_config
    n_text = n_vis = n_lora = 0
    for name, mod in model.named_modules():
        cls = type(mod).__name__
        if cls == "RotaryEmbedding" and hasattr(mod, "_compute_inv_freq"):
            mod.inv_freq.copy_(mod._compute_inv_freq(device=mod.inv_freq.device))
            assert getattr(mod, "scale_base", None) is None
            mod._seq_len_cached = 0
            n_text += 1
        elif cls == "VisionRotaryEmbeddingFast":
            half_head_dim = vc.width // (vc.width // vc.head_width) // 2
            hw = vc.image_size // vc.patch_size
            fresh = type(mod)(dim=half_head_dim, pt_seq_len=vc.pt_hw_seq_len,
                              ft_seq_len=hw if vc.intp_freq else None, patch_dropout=mod.patch_dropout)
            assert fresh.freqs_cos.shape == mod.freqs_cos.shape
            mod.freqs_cos.copy_(fresh.freqs_cos)
            mod.freqs_sin.copy_(fresh.freqs_sin)
            n_vis += 1
        if hasattr(mod, "lora_dropout_mask"):
            mod.lora_dropout_mask.fill_(1.0)
            n_lora += 1
    print(f"버퍼 복원: 텍스트 RoPE {n_text}개, 이미지 RoPE {n_vis}개, LoRA mask {n_lora}개")


model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True)
fix_buffers(model)
model = model.to(DEVICE).half().eval()

n_total = sum(p.numel() for p in model.parameters())
by_part = {}
for name, p in model.named_parameters():
    by_part[name.split(".")[0]] = by_part.get(name.split(".")[0], 0) + p.numel()
print(f"총 파라미터 {n_total / 1e6:.0f}M → fp16 용량 {n_total * 2 / 1e9:.2f} GB")
for k, v in sorted(by_part.items(), key=lambda x: -x[1]):
    print(f"  {k:<25s} {v / 1e6:8.1f}M  ({v * 2 / 1e9:.2f} GB)")


config.json:   0%|          | 0.00/2.15k [00:00<?, ?B/s]

configuration_clip.py:   0%|          | 0.00/12.6k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-clip-implementation:
- configuration_clip.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


modeling_clip.py:   0%|          | 0.00/25.5k [00:00<?, ?B/s]

eva_model.py:   0%|          | 0.00/26.5k [00:00<?, ?B/s]

rope_embeddings.py:   0%|          | 0.00/5.71k [00:00<?, ?B/s]

... (27 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/999 [00:00<?, ?it/s]

버퍼 복원: 텍스트 RoPE 24개, 이미지 RoPE 1개, LoRA mask 98개
총 파라미터 865M → fp16 용량 1.73 GB
  text_model                   560.9M  (1.12 GB)
  vision_model                 304.4M  (0.61 GB)
  logit_scale                    0.0M  (0.00 GB)


In [6]:
# 셀 6. 텍스트 토큰 길이 분포
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
all_texts = ds[TXT_COL]
lens = np.array([len(tokenizer(t, add_special_tokens=True)["input_ids"]) for t in all_texts])
MAX_LEN = 8192  # 모델 카드 기준 최대 입력 길이. tokenizer.model_max_length 도 함께 출력 # [확인 필요]
print(f"tokenizer.model_max_length = {tokenizer.model_max_length}")
print(f"토큰 길이 (전체 {len(lens)}개): 최대 {lens.max()} | 평균 {lens.mean():.1f} | 중앙값 {np.median(lens):.0f} | 95% {np.percentile(lens, 95):.0f}")
print(f"  {MAX_LEN} 토큰 초과(잘림) 비율: {(lens > MAX_LEN).mean() * 100:.2f}%")
# 참고용: 같은 문장이 SigLIP2(64), CLIP(77) 한도였다면 잘렸을 비율 (토크나이저가 달라 대략치)
for L in (64, 77):
    print(f"  (참고) {L} 토큰 초과 비율: {(lens > L).mean() * 100:.1f}%")


tokenizer_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

tokenizer.model_max_length = 8194
토큰 길이 (전체 4000개): 최대 98 | 평균 52.6 | 중앙값 51 | 95% 71
  8192 토큰 초과(잘림) 비율: 0.00%
  (참고) 64 토큰 초과 비율: 11.9%
  (참고) 77 토큰 초과 비율: 1.6%


In [7]:
# 셀 7. 이미지 임베딩 (속도 측정 포함, 저장/재사용)
from torch.utils.data import DataLoader, Dataset


class PILDataset(Dataset):
    """워커 프로세스에서 이미지 디코딩만 수행. 전처리(리사이즈/정규화)는 model.encode_image 가 담당."""

    def __init__(self, hf_ds):
        self.ds = hf_ds

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        return self.ds[i][IMG_COL].convert("RGB")


def to_tensor(x):
    return x if torch.is_tensor(x) else torch.from_numpy(np.asarray(x))


img_path = os.path.join(OUT, "val_img_emb.pt")
if os.path.exists(img_path):
    img_emb = torch.load(img_path)
    print("저장된 이미지 임베딩 로드:", tuple(img_emb.shape))
    speed = json.load(open(os.path.join(OUT, "speed.json")))
else:
    dl = DataLoader(PILDataset(val_ds), batch_size=IMG_BS, num_workers=NUM_WORKERS,
                    collate_fn=lambda b: b, prefetch_factor=4)
    feats, n_done, t_start = [], 0, None
    with torch.no_grad():
        for bi, batch in enumerate(dl):
            if bi == 1:  # 첫 배치는 워밍업(커널 컴파일 등)으로 제외
                torch.cuda.synchronize()
                t_start, n_done = time.time(), 0
            # 공식 API: encode_image(PIL 리스트) → 기본 L2 정규화된 numpy  # [확인 필요] 인자명
            e = model.encode_image(batch, batch_size=len(batch), normalize_embeddings=True)
            feats.append(to_tensor(e).float().cpu())
            n_done += len(batch)
    torch.cuda.synchronize()
    elapsed = time.time() - t_start
    ips = n_done / elapsed
    img_emb = torch.nn.functional.normalize(torch.cat(feats), dim=-1)  # 혹시 몰라 한 번 더 정규화
    assert torch.isfinite(img_emb).all(), "fp16 에서 NaN/Inf 발생 → DTYPE 을 float32 로 바꿔 다시 실행"
    torch.save(img_emb, img_path)
    np.save(os.path.join(OUT, "val_img_emb.npy"), img_emb.numpy())
    speed = {"images_per_sec_T4": ips, "est_16000_sec_T4": 16000 / ips}
    json.dump(speed, open(os.path.join(OUT, "speed.json"), "w"))
    print("이미지 임베딩:", tuple(img_emb.shape))

print(f"이미지 인코딩 속도 (T4, 디코딩 포함): {speed['images_per_sec_T4']:.1f} img/s")
print(f"→ 16,000장 추정: T4 {speed['est_16000_sec_T4'] / 60:.1f}분 | L4 대략 {speed['est_16000_sec_T4'] / 60 / 1.7:.1f}분 (L4≈T4×1.5~2 가정)")


preprocessor_config.json:   0%|          | 0.00/584 [00:00<?, ?B/s]

processing_clip.py:   0%|          | 0.00/2.67k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-clip-implementation:
- processing_clip.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


이미지 임베딩: (500, 1024)
이미지 인코딩 속도 (T4, 디코딩 포함): 7.3 img/s
→ 16,000장 추정: T4 36.5분 | L4 대략 21.5분 (L4≈T4×1.5~2 가정)


In [8]:
# 셀 8. 텍스트 임베딩 (task 어댑터별로 저장/재사용)
txt_emb = {}
for task in TEXT_TASKS:
    tag = task or "none"
    p = os.path.join(OUT, f"val_txt_emb_{tag}.pt")
    if os.path.exists(p):
        txt_emb[tag] = torch.load(p)
        print("저장된 텍스트 임베딩 로드:", tag)
        continue
    with torch.no_grad():
        # 공식 README: 검색 쿼리는 task='retrieval.query' 사용 # [확인 필요] task=None 일 때 동작
        e = model.encode_text(val_texts, task=task, batch_size=TXT_BS, normalize_embeddings=True)
    txt_emb[tag] = torch.nn.functional.normalize(to_tensor(e).float().cpu(), dim=-1)
    torch.save(txt_emb[tag], p)
    np.save(os.path.join(OUT, f"val_txt_emb_{tag}.npy"), txt_emb[tag].numpy())
    print("텍스트 임베딩:", tag, tuple(txt_emb[tag].shape))


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


텍스트 임베딩: retrieval.query (500, 1024)
텍스트 임베딩: none (500, 1024)


In [9]:
# 셀 9. zero-shot 평가 (텍스트 500개 → 이미지 500장, 정답은 같은 인덱스)
def evaluate(t, v):
    """t: [N,D] 텍스트, v: [N,D] 이미지 (둘 다 L2 정규화). 정답 = 대각선."""
    S = t @ v.T
    true = S.diag().unsqueeze(1)
    rank = 1 + (S > true).sum(dim=1)                # 정답 이미지 순위 r (1부터)
    r1 = (rank == 1).float().mean().item()
    ndcg = torch.where(rank <= 5, 1.0 / torch.log2(rank.float() + 1), torch.zeros_like(rank, dtype=torch.float)).mean().item()
    return {"recall@1": r1, "ndcg@5": ndcg, "score": 0.5 * (r1 + ndcg), "mean_rank": rank.float().mean().item()}


def trunc(x, d):
    return torch.nn.functional.normalize(x[:, :d], dim=-1)


results = []
for tag, t in txt_emb.items():
    for d in TRUNC_DIMS:
        r = evaluate(trunc(t, d), trunc(img_emb, d))
        results.append({"model": MODEL_ID, "text_task": tag, "dim": d, **{k: round(v, 4) for k, v in r.items()}})

print(f"{'text_task':<17s}{'dim':>6s}{'R@1':>8s}{'NDCG@5':>9s}{'score':>8s}{'meanRank':>10s}")
for r in results:
    print(f"{r['text_task']:<17s}{r['dim']:>6d}{r['recall@1']:>8.4f}{r['ndcg@5']:>9.4f}{r['score']:>8.4f}{r['mean_rank']:>10.2f}")

summary = {
    "model": MODEL_ID,
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
    "fp16_gb": round(n_total * 2 / 1e9, 3),
    "token_len": {"max": int(lens.max()), "mean": float(lens.mean()), "trunc_ratio": float((lens > MAX_LEN).mean())},
    "speed": speed,
    "results": results,
}
json.dump(summary, open(os.path.join(OUT, "summary.json"), "w"), indent=1, ensure_ascii=False)
print("\n저장:", os.path.join(OUT, "summary.json"))


text_task           dim     R@1   NDCG@5   score  meanRank
retrieval.query    1024  0.7860   0.9150  0.8505      1.26
retrieval.query     512  0.7860   0.9147  0.8504      1.27
retrieval.query     256  0.7940   0.9178  0.8559      1.26
none               1024  0.8080   0.9239  0.8659      1.24
none                512  0.7960   0.9192  0.8576      1.25
none                256  0.7880   0.9158  0.8519      1.26

저장: /kaggle/working/jina_clip_v2/summary.json


In [10]:
# 셀 10. (선택) 오프라인 제출용 내보내기 점검
# 채점 서버는 인터넷이 없을 수 있으므로 원격 코드(.py)까지 폴더 하나에 담겨야 합니다.
# save_pretrained 가 원격 코드 파일을 함께 복사하는지, 용량이 얼마인지 확인합니다. # [확인 필요]
DO_EXPORT = False
if DO_EXPORT:
    exp = os.path.join(OUT, "export")
    model.save_pretrained(exp)
    tokenizer.save_pretrained(exp)
    tot = 0
    for f in sorted(os.listdir(exp)):
        s = os.path.getsize(os.path.join(exp, f))
        tot += s
        print(f"  {f:<45s} {s / 1e6:9.1f} MB")
    print(f"합계 {tot / 1e9:.2f} GB")
    print(".py 파일:", [f for f in os.listdir(exp) if f.endswith(".py")])
    print("config auto_map:", json.load(open(os.path.join(exp, "config.json"))).get("auto_map"))
